# AgentReach AI — Phase 3 Safe Test: One Google Doc only

This notebook tests reading one non-sensitive Google Doc without listing your other files. It searches for the exact title **AgentReach Test Document** only, then lets you select one matching result, requires an explicit confirmation phrase, and exposes a Needle tool restricted to the exact selected file ID.

## Before you start
1. In Google Drive, create a new Google Doc named exactly `AgentReach Test Document`.
2. Put only this text inside: `This is a non-sensitive test document for AgentReach AI. The integration is working.`
3. Do not put personal information, account details, passwords, or tokens in it.
4. Run these cells from top to bottom. Google authorization may grant notebook code broader access than this tool uses, so review the code and any Google consent prompt carefully.

This notebook does not list unrelated files, does not read PDFs/Sheets, and does not create, edit, share, move, or delete any files. If the exact title is not found, it will tell you and the selection/read steps will be skipped.

In [ ]:
%pip install -q cactus-needle google-api-python-client

## 2. Import dependencies

In [ ]:
import json
import sys
import needle
from google.colab import auth
from googleapiclient.discovery import build

print('Python:', sys.version.split()[0])
print('Needle import: OK')

## 3. Initialize Google Drive access

If Google asks for authorization, review the requested access before approving. Do not continue if you are uncomfortable with the permission prompt.

In [ ]:
auth.authenticate_user()
drive_service = build('drive', 'v3')
print('Drive client initialized.')

## 4. Search for the exact test title only

This query is fixed to the exact title `AgentReach Test Document`. It will not print unrelated personal document names.

In [ ]:
TEST_DOC_NAME = 'AgentReach Test Document'
GOOGLE_DOCS_MIME = 'application/vnd.google-apps.document'
query = f"name = '{TEST_DOC_NAME}' and mimeType = '{GOOGLE_DOCS_MIME}' and trashed = false"
response = drive_service.files().list(
    q=query,
    pageSize=5,
    orderBy='modifiedTime desc',
    fields='files(id,name,mimeType,modifiedTime)',
).execute()
test_docs = response.get('files', [])
print('Exact-title test documents found:', len(test_docs))
if not test_docs:
    print('No document with that exact title was found. Create it in Drive, then rerun this cell and the cells below it.')
else:
    for index, doc in enumerate(test_docs, start=1):
        print(f"{index}. {doc.get('name')} | ID: {doc['id']} | Modified: {doc.get('modifiedTime', 'unknown')}")
    print('Only documents with the exact test title are shown.')

## 5. Select and explicitly approve one test document

Choose a number from the exact-title results above, not an ID from your other files.

In [ ]:
approved_file_id = None
approved_to_read = False
if not test_docs:
    print('Skipped: no exact-title test document is available.')
else:
    choice = input("Enter the number of the AgentReach Test Document to use, or type SKIP: ").strip()
    if choice.upper() == 'SKIP':
        print('Skipped. No content will be read.')
    elif choice.isdigit() and 1 <= int(choice) <= len(test_docs):
        chosen_doc = test_docs[int(choice) - 1]
        print('Selected:', chosen_doc.get('name'))
        phrase = input("To approve reading this test document, type READ TEST DOCUMENT: ").strip()
        approved_to_read = (phrase == 'READ TEST DOCUMENT')
        approved_file_id = chosen_doc['id'] if approved_to_read else None
        print('Approved for this exact file only.' if approved_to_read else 'Not approved. No content will be read.')
    else:
        print('Invalid choice. No content will be read.')

## 6. Define the restricted preview tool

This tool only accepts the approved file ID, verifies that it is a native Google Doc, and returns at most 5,000 characters.

In [ ]:
@needle.tool(triggers=[r'(?i)\bpreview approved test doc\b', r'(?i)\bread the approved test document\b'])
def preview_approved_test_doc(file_id: str) -> dict:
    """Preview up to 5,000 characters from the single Google Doc titled AgentReach Test Document that the user explicitly selected and approved in this notebook. Requires its exact file ID."""
    if not approved_to_read or not approved_file_id:
        return {'ok': False, 'error': 'No test document has been explicitly approved for reading.'}
    if not isinstance(file_id, str) or file_id != approved_file_id:
        return {'ok': False, 'error': 'Access denied: only the exact user-approved test document is allowed.'}
    try:
        meta = drive_service.files().get(
            fileId=approved_file_id,
            fields='id,name,mimeType',
        ).execute()
        if meta.get('mimeType') != GOOGLE_DOCS_MIME or meta.get('name') != TEST_DOC_NAME:
            return {'ok': False, 'error': 'Only the exact-title native Google test document can be previewed.'}
        raw = drive_service.files().export(
            fileId=approved_file_id,
            mimeType='text/plain',
        ).execute()
        text = raw.decode('utf-8', errors='replace') if isinstance(raw, bytes) else str(raw)
        limit = 5000
        return {
            'ok': True,
            'name': meta.get('name'),
            'preview_characters': min(len(text), limit),
            'truncated': len(text) > limit,
            'preview': text[:limit],
            'note': 'Read-only preview of the explicitly approved test document.',
        }
    except Exception as exc:
        return {'ok': False, 'error': f'Test document preview failed: {type(exc).__name__}'}

print('Preview tool defined. Test document approved:', bool(approved_file_id))

## 7. Test the Google Docs API directly

In [ ]:
if not approved_to_read or not approved_file_id:
    print('Skipped because no test document was approved.')
else:
    direct_result = preview_approved_test_doc(approved_file_id)
    print(json.dumps(direct_result, indent=2, ensure_ascii=False, default=str))
    assert direct_result.get('ok') is True, 'Direct preview failed. Inspect the safe error above.'

## 8. Test Needle 3 tool calling

In [ ]:
if not approved_to_read or not approved_file_id:
    print('Skipped because no test document was approved.')
else:
    agent = needle.Needle(tools=[preview_approved_test_doc])
    result = agent.run(
        'Preview approved test doc using preview_approved_test_doc. The approved file ID is ' + approved_file_id +
        '. Call the tool only for this exact approved test document.'
    )
    print(json.dumps(result, indent=2, ensure_ascii=False, default=str))

## Success checkpoint

- The search shows only the exact-title test document(s), not unrelated files.
- You explicitly select one result and approve with `READ TEST DOCUMENT`.
- The direct API preview returns `ok: true`.
- The Needle result shows a tool call/result for the same test document.

If the exact-title search finds nothing, create the test document in Drive and rerun the search and following cells. If the direct preview succeeds but Needle does not call the tool, the Drive integration works and the remaining issue is tool selection, not Google authorization. No writes or deletes are included in this notebook.